In [9]:
from glob import glob
import pandas as pd
import os
from tqdm import tqdm as tqdm 
from IPython.display import Image
from pathlib import Path
# import imageio
# import moviepy.video.io.ImageSequenceClip
from tqdm import tqdm
import h5py
import math
import matplotlib
# matplotlib.use('Agg') # Must be before importing matplotlib.pyplot or pylab!
import matplotlib.pyplot as plt
import copy
import numpy as np
import os
import json
import scipy.stats
import time
from types import SimpleNamespace
import random
import pandas as pd
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns

/afs/csail.mit.edu/u/c/czw/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-81mzfwmn because there was an issue with the default path (/afs/csail.mit.edu/u/c/czw/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [86]:
!pwd

/storage/czw/neuroprobe


In [78]:
NEUROPROBE_TASKS_MAPPING = {
    'onset': 'Sentence Onset',
    'speech': 'Speech',
    'volume': 'Volume', 
    'delta_volume': 'Delta Volume',
    'pitch': 'Voice Pitch',

    'word_index': 'Word Position',
    'word_gap': 'Inter-word Gap',
    'gpt2_surprisal': 'GPT-2 Surprisal',
    'word_head_pos': 'Head Word Position',
    'word_part_speech': 'Part of Speech',

    'word_length': 'Word Length',
    'global_flow': 'Global Optical Flow',
    'local_flow': 'Local Optical Flow',
    'frame_brightness': 'Frame Brightness',
    'face_num': 'Number of Faces',
}

In [1]:
!pwd

/storage/czw/neuroprobe


In [51]:
!unzip  'run_all_mlp_no_sort_1s.zip'

Archive:  run_all_mlp_no_sort_1s.zip
   creating: home/geeling/Projects/tb_buildathon/torch_brain/examples/neuroprobe_eval/outputs/run_all_mlp_no_sort_1s/
   creating: home/geeling/Projects/tb_buildathon/torch_brain/examples/neuroprobe_eval/outputs/run_all_mlp_no_sort_1s/mlp_laplacian_stft/
   creating: home/geeling/Projects/tb_buildathon/torch_brain/examples/neuroprobe_eval/outputs/run_all_mlp_no_sort_1s/mlp_laplacian_stft/Within-Session/
   creating: home/geeling/Projects/tb_buildathon/torch_brain/examples/neuroprobe_eval/outputs/run_all_mlp_no_sort_1s/mlp_laplacian_stft/Within-Session/word_gap/
   creating: home/geeling/Projects/tb_buildathon/torch_brain/examples/neuroprobe_eval/outputs/run_all_mlp_no_sort_1s/mlp_laplacian_stft/Within-Session/word_gap/sub1_trial1/
   creating: home/geeling/Projects/tb_buildathon/torch_brain/examples/neuroprobe_eval/outputs/run_all_mlp_no_sort_1s/mlp_laplacian_stft/Within-Session/word_gap/sub1_trial1/wandb/
   creating: home/geeling/Projects/tb_build

In [52]:
!mv home geeling_rebuttal_results_mlp

In [70]:
cat geeling_rebuttal_results_mlp/geeling/Projects/tb_buildathon/torch_brain/examples/neuroprobe_eval/outputs/run_all_mlp_no_sort_1s/mlp_laplacian_stft/Within-Session/delta_volume/sub10_trial0/eval_results/Within-Session/mlp_laplacian_stft/population_btbank10_0_delta_volume.json

{
    "model_name": "MLP",
    "author": "Your Name",
    "description": "Simple MLP using all electrodes (laplacian_stft).",
    "organization": "Your Organization",
    "organization_url": "https://your-url.com",
    "timestamp": 1763690813.1138535,
    "evaluation_results": {
        "btbank10_0": {
            "population": {
                "one_second_after_onset": {
                    "time_bin_start": 0.0,
                    "time_bin_end": 1.0,
                    "folds": [
                        {
                            "train_accuracy": 1.0,
                            "train_roc_auc": 1.0,
                            "val_accuracy": 0.6674285714285715,
                            "val_roc_auc": 0.7228613523087051,
                            "test_accuracy": 0.6685714285714286,
                            "test_roc_auc": 0.7139692590618893,
                            "fold_idx": 0
                        },
                        {
                            "tr

In [71]:


def get_test_results(results):
    eval_results = results["evaluation_results"]
    assert len(eval_results.keys()) == 1
    records = []
    for subject_trial in eval_results:
        subject = subject_trial.split("_")[0][len("btbank"):]
        trial = subject_trial.split("_")[1]
        # subject_trial_results = eval_results[subject_trial]['electrode']
        subject_trial_results = eval_results[subject_trial]['population']
        # for electrode in subject_trial_results:
        electrode_results = subject_trial_results
        time_bin_results = electrode_results["one_second_after_onset"]
        # for time_bin in time_bin_results:
        # time_bin_start = time_bin['one_second_after_onset']
        fold_results = time_bin_results['folds']
        avg_test = np.mean([f['test_roc_auc'] for f in fold_results])
        records.append({
            "subject": subject,
            "trial": trial,
            "ID": f'sub_{subject}',
            # "electrode": electrode,
            "avg_test": avg_test,
            # "time_bin": time_bin_start,
            "task": eval_name,
            "model": results["model_name"]
        })
        return records



In [72]:
all_records = []
results_paths = glob("geeling_rebuttal_results/geeling/Projects/tb_buildathon/torch_brain/examples/neuroprobe_eval/outputs/run_all_cnn_no_sort_1s/cnn_laplacian_stft/Within-Session/*/*/eval_results/Within-Session/cnn_laplacian_stft/*.json")
for results_path in results_paths:
    name = Path(results_path).stem
    eval_name = "_".join(name.split("_")[3:])
    with open(results_path, "r") as f:
        data = json.load(f)
    all_records += get_test_results(data)

results_paths = glob("geeling_rebuttal_results_mlp/geeling/Projects/tb_buildathon/torch_brain/examples/neuroprobe_eval/outputs/run_all_mlp_no_sort_1s/mlp_laplacian_stft/Within-Session/*/*/eval_results/Within-Session/mlp_laplacian_stft/*.json")
for results_path in results_paths:
    name = Path(results_path).stem
    eval_name = "_".join(name.split("_")[3:])
    with open(results_path, "r") as f:
        data = json.load(f)
    all_records += get_test_results(data)

In [82]:
results_df = pd.DataFrame.from_records(all_records)

results_df.task = [NEUROPROBE_TASKS_MAPPING[x] for x in results_df.task]
mean_df = results_df.pivot_table(index=["task"], columns="model", values=["avg_test"], aggfunc="mean")
mean_df.columns = mean_df.columns.get_level_values(1)
std_df = results_df.pivot_table(index=["task"], columns="model", values=["avg_test"], aggfunc="std")
std_df.columns = std_df.columns.get_level_values(1)


In [83]:
results_df

,subject,trial,ID,avg_test,task,model
0,3,0,sub_3,0.772975,Global Optical Flow,CNN
1,7,0,sub_7,0.630095,Global Optical Flow,CNN
2,4,0,sub_4,0.588358,Global Optical Flow,CNN
3,1,1,sub_1,0.667312,Global Optical Flow,CNN
4,1,2,sub_1,0.684258,Global Optical Flow,CNN
...,...,...,...,...,...,...
355,7,1,sub_7,0.592625,Local Optical Flow,MLP
356,10,0,sub_10,0.572589,Local Optical Flow,MLP
357,3,1,sub_3,0.764937,Local Optical Flow,MLP
358,4,1,sub_4,0.603744,Local Optical Flow,MLP


In [74]:
# mean_df = df.groupby("task")['avg_test'].mean()
# std_df = df.groupby("task")['avg_test'].std()

In [85]:
print((mean_df.round(2).astype(str) + "±" + std_df.round(2).astype(str)).to_markdown())

| task                | CNN       | MLP       |
|:--------------------|:----------|:----------|
| Delta Volume        | 0.76±0.09 | 0.75±0.1  |
| Frame Brightness    | 0.53±0.07 | 0.51±0.07 |
| GPT-2 Surprisal     | 0.6±0.06  | 0.61±0.06 |
| Global Optical Flow | 0.64±0.06 | 0.63±0.05 |
| Head Word Position  | 0.61±0.05 | 0.61±0.05 |
| Inter-word Gap      | 0.58±0.06 | 0.6±0.05  |
| Local Optical Flow  | 0.63±0.05 | 0.62±0.06 |
| Number of Faces     | 0.54±0.04 | 0.52±0.06 |
| Part of Speech      | 0.62±0.08 | 0.59±0.06 |
| Sentence Onset      | 0.89±0.05 | 0.89±0.07 |
| Speech              | 0.9±0.05  | 0.88±0.07 |
| Voice Pitch         | 0.61±0.07 | 0.59±0.06 |
| Volume              | 0.75±0.14 | 0.73±0.12 |
| Word Length         | 0.64±0.07 | 0.61±0.07 |
| Word Position       | 0.73±0.1  | 0.73±0.11 |


In [76]:
mean_df = df.groupby("task")['avg_test'].std()

In [49]:
a = 0.61683  
f"{a:.2f} \pm "

'0.62'

In [50]:
pwd

'/storage/czw/neuroprobe'